# C2.7 · Benchmark design and critique

**Function C — Offensive Security & Research → The Security Researcher**  ·  *AI for Security*

Builds on **[C2.6 · Building the research harness](https://spbreed.github.io/cyber-commons/lessons/C2.6.html)**.

| | |
|---|---|
| Open-source tooling | Cyber Commons eval harness |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Benchmark critique is a research skill, and the three questions that decide
whether a security-harness number means anything are always the same:

1. **What is the class balance?** If one class dominates, a constant answer
   scores well. Always compute what always-guessing-the-majority scores, and
   report the *lift over that baseline* rather than the raw number.
2. **Is the key held out?** If the harness has seen the answers — through
   training, through prompt examples, or through its own logs — the number is a
   training metric.
3. **How are files matched?** Bare-basename matching on corpora that reuse
   filenames turns accuracy into a partly random variable.

C1.6 attacked a benchmark. This lesson designs one that survives the attack.

## 2 · Demo — the three checks, applied

In [ ]:
import json
from collections import Counter
from dataclasses import dataclass

@dataclass
class Truth:
    qid: str; cwe: str; file: str

def make(n, classes, files_collide=False):
    t = {}
    for i in range(1, n + 1):
        cwe = classes[i % len(classes)]
        fname = f"{i}.py" if files_collide else f"{cwe}/{i}.py"
        t[f"q{i}"] = Truth(f"q{i}", cwe, fname if files_collide else f"{cwe}/{i}.py")
    return t

def path_key(p):
    parts = [x for x in p.replace("\\", "/").split("/") if x not in ("", ".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")
def basename(p): return p.replace("\\", "/").split("/")[-1]

def score(answers, truths, matcher=path_key):
    conf = expert = 0
    for q, t in truths.items():
        try: d = json.loads(answers[q])
        except (json.JSONDecodeError, KeyError): continue
        conf += 1
        if matcher(d["file"]) != matcher(t.file): continue
        expert += 1.0 if d["cwe"].upper() == t.cwe else 0.5
    return {"conformance": round(conf/len(truths), 3),
            "expert": round(expert/len(truths), 3)}

def majority_baseline(truths, matcher=path_key):
    maj = Counter(t.cwe for t in truths.values()).most_common(1)[0][0]
    ans = {q: json.dumps({"qid": q, "cwe": maj, "file": t.file, "rationale": "x"})
           for q, t in truths.items()}
    return score(ans, truths, matcher)["expert"], maj

SKEWED   = make(40, ["CWE-89"]*7 + ["CWE-78"])
BALANCED = make(40, ["CWE-89", "CWE-78", "CWE-22", "CWE-798"])
for name, t in (("skewed", SKEWED), ("balanced", BALANCED)):
    counts = Counter(x.cwe for x in t.values())
    floor, maj = majority_baseline(t)
    print(f"{name:10s} balance={dict(counts)}")
    print(f"{'':10s} always-guess-{maj} scores {floor:.3f}  ← the floor to beat")

## 3 · Check 2 and 3 — held-out keys and file matching

In [ ]:
# a harness that has seen the key vs one that has not
def harness(truths, seen_key, skill=0.6, seed=3):
    import random
    rng = random.Random(seed)
    ans = {}
    for q, t in truths.items():
        if seen_key or rng.random() < skill:
            cwe, f = t.cwe, t.file
        else:
            cwe, f = "CWE-89", t.file
        ans[q] = json.dumps({"qid": q, "cwe": cwe, "file": f, "rationale": "untrusted input"})
    return ans

for label, seen in (("key held out", False), ("key leaked into the harness", True)):
    s = score(harness(BALANCED, seen), BALANCED)
    floor, _ = majority_baseline(BALANCED)
    print(f"{label:30s} expert={s['expert']:.3f}  lift over floor={s['expert']-floor:+.3f}")

# file matching
WRONG_DIR = {q: json.dumps({"qid": q, "cwe": t.cwe, "file": f"CWE-89/{q[1:]}.py",
                            "rationale": "x"})
             for q, t in BALANCED.items()}
print()
for m, name in ((path_key, "path_key"), (basename, "basename only")):
    print(f"answers naming the wrong directory, matched by {name:14s}: "
          f"expert={score(WRONG_DIR, BALANCED, m)['expert']:.3f}")

## 4 · The control — a benchmark spec that survives critique

In [ ]:
def benchmark_spec(truths, answers, matcher=path_key, key_held_out=True):
    counts = Counter(t.cwe for t in truths.values())
    floor, maj = majority_baseline(truths, matcher)
    s = score(answers, truths, matcher)
    collisions = len(truths) - len({matcher(t.file) for t in truths.values()})
    return {
      "n": len(truths),
      "class_balance": dict(counts),
      "majority_class": maj,
      "trivial_baseline": floor,
      "conformance": s["conformance"],
      "expert_accuracy": s["expert"],
      "LIFT_over_baseline": round(s["expert"] - floor, 3),
      "key_held_out": key_held_out,
      "matcher": matcher.__name__,
      "matcher_collisions": collisions,
      "publishable": (key_held_out and collisions == 0
                      and s["expert"] - floor > 0.1),
    }

good = benchmark_spec(BALANCED, harness(BALANCED, False), path_key, True)
bad  = benchmark_spec(SKEWED,   harness(SKEWED, True),    basename, False)
for label, spec in (("designed properly", good), ("as usually published", bad)):
    print(f"=== {label} ===")
    for k, v in spec.items(): print(f"   {k:22s} {v}")
    print()
assert good["publishable"] and not bad["publishable"]

## What you just proved

The skewed corpus gives always-guessing-the-majority about 0.88; the balanced one about 0.31. A harness with a leaked key scores near 1.0 while the held-out one lands near its true skill. Wrong-directory answers score near 0 under `path_key` and near 1.0 under basename matching. The properly designed spec is marked publishable and the usual one is not.

## Your turn

Apply the three checks to one published agentic-security benchmark you rely on. Write the critique as a repro card so someone can check your claim rather than taking it on trust.

---

**Next → [C2.8 · From finding to control](https://spbreed.github.io/cyber-commons/lessons/C2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*